In [1]:
import json
import os
import time
import pytz
import requests
from pathlib import Path
from datetime import datetime,timezone,timedelta
from scipy.optimize import curve_fit
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
from dotenv import load_dotenv
from bot_template import BaseBot, OrderBook, Order, OrderRequest, OrderResponse, Trade, Side, Product

load_dotenv()

EXCHANGE_URL  = "http://ec2-52-49-69-152.eu-west-1.compute.amazonaws.com/"   
USERNAME = os.getenv("EXCHANGE_USERNAME")
PASSWORD = os.getenv("EXCHANGE_PASSWORD")

LONDON_TZ = pytz.timezone('Europe/London')

In [2]:
LONDON_LAT, LONDON_LON = 51.5074, -0.1278

def get_weather(past_steps=96, forecast_steps=96):
    """伦敦15分钟天气预报, 返回一个包含以下列的DataFrame: 时间, 温度, 湿度."""
    variables = "temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,wind_speed_10m,cloud_cover,visibility"
    resp = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": LONDON_LAT, "longitude": LONDON_LON,
        "minutely_15": variables,
        "past_minutely_15": past_steps,
        "forecast_minutely_15": forecast_steps,
        "timezone": "Europe/London",
    })
    resp.raise_for_status()
    m = resp.json()["minutely_15"]
    return pd.DataFrame({
        "time": pd.to_datetime(m["time"]).tz_localize("Europe/London"),
        "temperature": m["temperature_2m"],
        "humidity": m["relative_humidity_2m"],
    })

In [3]:
def get_wx_spot():
    weather_df = get_weather()
    weather_df = weather_df[(weather_df["time"]<=LONDON_TZ.localize(datetime(2026,3,1,12,0,0)))&(weather_df["time"]>=LONDON_TZ.localize(datetime(2026,2,28,12,0,0)))]
    weather_df["temperature"] = weather_df["temperature"]*9/5+32
    wx_spot = weather_df[weather_df["time"]==LONDON_TZ.localize(datetime(2026,3,1,12,0,0))]["temperature"] * weather_df[weather_df["time"]==LONDON_TZ.localize(datetime(2026,3,1,12,0,0))]["humidity"]
    return round(wx_spot.item())

def get_wx_sum():
    weather_df = get_weather()
    weather_df = weather_df[(weather_df["time"]<=LONDON_TZ.localize(datetime(2026,3,1,12,0,0)))&(weather_df["time"]>=LONDON_TZ.localize(datetime(2026,2,28,12,0,0)))]
    weather_df["temperature"] = weather_df["temperature"]*9/5+32
    weather_df["temp_hum_product"] = weather_df["temperature"]*weather_df["humidity"]
    wx_sum = weather_df["temp_hum_product"].sum()/100
    return round(wx_sum.item())

In [4]:
THAMES_MEASURE = "0006-level-tidal_level-i-15_min-mAOD"

def get_thames(limit=400):
    """获取威斯敏斯特地区泰晤士河的最新潮汐数据. 返回一个包含以下列的DataFrame: 时间, 水平"""
    resp = requests.get(
        f"https://environment.data.gov.uk/flood-monitoring/id/measures/{THAMES_MEASURE}/readings",
        params={"_sorted": "", "_limit": limit},
    )
    resp.raise_for_status()
    items = resp.json().get("items", [])
    df = pd.DataFrame(items)[["dateTime", "value"]].rename(columns={"dateTime": "time", "value": "level"})
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert("Europe/London")
    return df.sort_values("time").reset_index(drop=True)

In [5]:
def sin_function(x,a,b,c,d):
    """正弦拟合函数y=a+bsin(cx+d)"""
    return a+b*np.sin(c*x+d)

def fit_thames_data(thames_df):
    """用正弦函数y=a+bsin(cx+d)拟合水位数据, 返回拟合参数(a,b,c,d)"""
    start_time = thames_df["time"].min()
    hours_since_start = (thames_df["time"]-start_time).dt.total_seconds()/3600
    x_data = hours_since_start.values
    y_data = thames_df["level"].values
    initial_guess = [np.mean(y_data),np.std(y_data),2*np.pi/12,0] 
    popt, _ = curve_fit(sin_function,x_data,y_data,p0=initial_guess,maxfev=5000,)
    a,b,c,d = popt
    return a,b,c,d


In [6]:
def get_tide_spot():
    thames_df = get_thames()
    a,b,c,d = fit_thames_data(thames_df)
    predicted_level = sin_function((LONDON_TZ.localize(datetime(2026,3,1,12,0,0))-thames_df["time"].min()).total_seconds()/3600,a,b,c,d)
    tide_spot = abs(predicted_level)*1000
    return round(tide_spot.item())

def get_tide_swing():
    thames_df = get_thames()
    thames_df = thames_df[thames_df["time"]>=LONDON_TZ.localize(datetime(2026,2,28,12,0,0))]
    a,b,c,d = fit_thames_data(thames_df)
    
    date_range = pd.date_range(start=max(thames_df['time'])+timedelta(minutes=15),end=LONDON_TZ.localize(datetime(2026,3,1,12,0,0)),freq="15min",tz="Europe/London")
    thames_missing_df = pd.DataFrame({'time':date_range})
    hours_since_start = (thames_missing_df["time"]-thames_df["time"].min()).dt.total_seconds()/3600
    thames_missing_df["level"] = a+b*np.sin(c*hours_since_start.values+d)
    thames_df = pd.concat([thames_df,thames_missing_df], ignore_index=True).sort_values("time").drop_duplicates(subset=["time"], keep="last").reset_index(drop=True)

    thames_df["level_cm"] = round(thames_df["level"]*100)
    thames_df["diff_cm"] = thames_df["level_cm"].diff().abs().dropna().reset_index(drop=True)
    thames_df["put_payoff"] = np.maximum(0,20-thames_df["diff_cm"])
    thames_df["call_payoff"] = np.maximum(0,thames_df["diff_cm"]-25)
    thames_df["total_payoff"] = thames_df["put_payoff"]+thames_df["call_payoff"]

    return thames_df["total_payoff"].sum()


In [7]:
AERODATABOX_KEY = os.getenv("AERODATABOX_KEY")
AERODATABOX_HOST = "aerodatabox.p.rapidapi.com"
AIRPORT = "LHR"

def fetch_flights(airport=AIRPORT,offset_minutes=-360,duration_minutes=720,filters:dict|None=None):
    """按相对时间窗口(从现在开始的偏移量)获取航班信息. 该API抓取次数有限制所以说要保留历史文档"""
    params = f"?offsetMinutes={offset_minutes}&durationMinutes={duration_minutes}&direction=Both"
    if filters:
        for k, v in filters.items():
            params += f"&{k}={'true' if v else 'false'}"
    url = f"https://{AERODATABOX_HOST}/flights/airports/iata/{airport}{params}"
    resp = requests.get(url, headers={
        "x-rapidapi-host": AERODATABOX_HOST, "x-rapidapi-key": AERODATABOX_KEY})
    resp.raise_for_status()
    data = json.loads(resp.text)
    return data

def fetch_flights_range(airport=AIRPORT,from_local="2026-02-28T12:00",to_local="2026-02-29T00:00",filters: dict|None = None):
    """根据明确的本地时间范围(最大跨度为12小时)获取航班信息. 该API抓取次数有限制所以说要保留历史文档"""
    params = "?direction=Both"
    if filters:
        for k, v in filters.items():
            params += f"&{k}={'true' if v else 'false'}"
    url = f"https://{AERODATABOX_HOST}/flights/airports/iata/{airport}/{from_local}/{to_local}{params}"
    resp = requests.get(url, headers={
        "x-rapidapi-host": AERODATABOX_HOST, "x-rapidapi-key": AERODATABOX_KEY})
    resp.raise_for_status()
    data = json.loads(resp.text)
    return data

In [8]:
flights_data_1 = fetch_flights_range(from_local="2026-02-28T12:00",to_local="2026-03-01T00:00")

In [9]:
flights_data_2 = fetch_flights_range(from_local="2026-03-01T00:00",to_local="2026-03-01T12:00")

In [10]:
def get_lhr_count():
    global flights_data_1, flights_data_2    
    lhr_count = len(flights_data_1.get('arrivals',[]))+len(flights_data_1.get('departures',[]))+len(flights_data_2.get('arrivals',[]))+len(flights_data_2.get('departures',[]))
    return lhr_count

In [11]:
def flight_time(flight_record,flight_type):
    """记录航班的时间. 先用实际落地/起飞时间, 否则用计划时间"""
    time_fields = ["runwayTime","revisedTime","scheduledTime"]
    movement = flight_record["movement"]
    for field in time_fields:
        if field in movement and movement[field].get("utc"):
            flight_time = movement[field]["utc"]
            break
    flight_time = pytz.utc.localize(datetime.strptime(flight_time,"%Y-%m-%d %H:%MZ")).astimezone(LONDON_TZ)
    return round(flight_time)

def group_flights(flights_data,start_time,end_time):
    """按30分钟区间分组统计到达/出发航班数"""
    arrival_times = []
    for arr in flights_data.get("arrivals",[]):
        try:
            arr_time = flight_time(arr,"arrival")
            if start_time<=arr_time<end_time:
                arrival_times.append(arr_time)
        except:
            pass
    departure_times = []
    for dep in flights_data.get("departures", []):
        try:
            dep_time = flight_time(dep, "departure")
            if start_time<=dep_time<end_time: 
                departure_times.append(dep_time)
        except:
            pass
    return (len(arrival_times),len(departure_times))

def get_lhr_index():
    global flights_data_1, flights_data_2
    lhr_index = 0
    date_range = pd.date_range(start=LONDON_TZ.localize(datetime(2026,2,28,12,0,0)),end=LONDON_TZ.localize(datetime(2026,3,1,12,0,0)),freq="30min",tz="Europe/London")
    for index in range(24):
        arr,dep = group_flights(flights_data_1,date_range[index],date_range[index+1])
        lhr_index += ((arr-dep)/(arr+dep))*100 if arr+dep else 0
    for index in range(24):
        arr,dep = group_flights(flights_data_2,date_range[index+24],date_range[index+25])
        lhr_index += ((arr-dep)/(arr+dep))*100 if arr+dep else 0
    lhr_index = abs(lhr_index)
    return round(lhr_index)


In [12]:
def get_settlement_price(symbol):
    price_map = {
        "TIDE_SPOT": get_tide_spot,
        "TIDE_SWING": get_tide_swing,
        "WX_SPOT": get_wx_spot,
        "WX_SUM": get_wx_sum,
        "LHR_COUNT": get_lhr_count,
        "LHR_INDEX": get_lhr_index,
        "LON_ETF": get_lon_etf,
        "LON_FLY": get_lon_fly,
    }
    fn = price_map.get(symbol)
    return fn() if fn else 10.0

def get_lon_etf():
    return get_tide_spot()+get_wx_spot()+get_lhr_count()

def get_lon_fly():
    etf_price = get_lon_etf() 
    part1 = 2*max(6200-etf_price,0)
    part2 = max(etf_price-6200,0)
    part3 = -2*max(etf_price-6600,0)
    part4 = 3*max(etf_price-7000,0)
    return part1 + part2 + part3 + part4

In [ ]:
class FairValueEngine:
    """
    维护每个产品的 fair value 点估计和不确定性(std).
    数据缓存避免重复 API 调用.
    """
    CACHE_TTL = 60  # 天气/潮汐缓存60秒

    def __init__(self):
        self._weather_cache = None
        self._weather_ts = 0
        self._thames_cache = None
        self._thames_ts = 0
        self._flights_cache = None  # 航班数据只抓一次

    # ── 数据层 ────────────────────────────────────────────────────────────────
    def _get_weather_cached(self):
        if time.time() - self._weather_ts > self.CACHE_TTL:
            self._weather_cache = get_weather(past_steps=96, forecast_steps=96)
            self._weather_ts = time.time()
        return self._weather_cache

    def _get_thames_cached(self):
        if time.time() - self._thames_ts > self.CACHE_TTL:
            self._thames_cache = get_thames(limit=672)  # ~7天
            self._thames_ts = time.time()
        return self._thames_cache

    def init_flights(self):
        """比赛开始时调用一次，抓取完整航班数据"""
        if self._flights_cache is None:
            d1 = fetch_flights_range(from_local='2026-02-28T12:00', to_local='2026-03-01T00:00')
            d2 = fetch_flights_range(from_local='2026-03-01T00:00', to_local='2026-03-01T12:00')
            self._flights_cache = (d1, d2)
            print(f'Flights loaded: {get_lhr_count_from(d1, d2)} total')
        return self._flights_cache

    # ── 各产品 fair value ─────────────────────────────────────────────────────
    def wx_spot_fv(self):
        df = self._get_weather_cached()
        settle = LONDON_TZ.localize(datetime(2026, 3, 1, 12, 0, 0))
        df = df[(df['time'] >= LONDON_TZ.localize(datetime(2026, 2, 28, 12, 0, 0))) &
                (df['time'] <= settle)].copy()
        df['temp_f'] = df['temperature'] * 9/5 + 32
        row = df[df['time'] == settle]
        if row.empty:
            row = df.iloc[[-1]]
        fv = float(row['temp_f'].values[0] * row['humidity'].values[0])
        # 不确定性：用过去24h的 temp*hum 标准差作为代理
        std = float((df['temp_f'] * df['humidity']).std()) if len(df) > 1 else fv * 0.02
        return round(fv), max(std, 1.0)

    def wx_sum_fv(self):
        df = self._get_weather_cached()
        settle = LONDON_TZ.localize(datetime(2026, 3, 1, 12, 0, 0))
        df = df[(df['time'] >= LONDON_TZ.localize(datetime(2026, 2, 28, 12, 0, 0))) &
                (df['time'] <= settle)].copy()
        df['temp_f'] = df['temperature'] * 9/5 + 32
        df['prod'] = df['temp_f'] * df['humidity']
        # 已实测部分 + 预报部分
        now = LONDON_TZ.localize(datetime.now(timezone.utc).replace(tzinfo=None))
        observed = df[df['time'] <= now]['prod'].sum() / 100
        forecast = df[df['time'] > now]['prod'].sum() / 100
        fv = observed + forecast
        # 不确定性只来自预报部分（已实测的方差为0）
        n_forecast = len(df[df['time'] > now])
        per_step_std = float(df['prod'].std()) / 100 if len(df) > 1 else fv * 0.01
        std = per_step_std * (n_forecast ** 0.5)
        return round(fv), max(std, 1.0)

    def tide_spot_fv(self):
        df = self._get_thames_cached()
        a, b, c, d = fit_thames_data(df)
        settle_h = (LONDON_TZ.localize(datetime(2026, 3, 1, 12, 0, 0)) - df['time'].min()).total_seconds() / 3600
        fv = abs(sin_function(settle_h, a, b, c, d)) * 1000
        # 不确定性：拟合残差的 std * 1000
        hours = (df['time'] - df['time'].min()).dt.total_seconds() / 3600
        residuals = df['level'].values - sin_function(hours.values, a, b, c, d)
        std = float(np.std(residuals)) * 1000
        return round(fv), max(std, 1.0)

    def tide_swing_fv(self):
        df = self._get_thames_cached()
        df = df[df['time'] >= LONDON_TZ.localize(datetime(2026, 2, 28, 12, 0, 0))].copy()
        a, b, c, d = fit_thames_data(df)
        # 补全到结算时间
        end = LONDON_TZ.localize(datetime(2026, 3, 1, 12, 0, 0))
        dr = pd.date_range(start=df['time'].max() + timedelta(minutes=15), end=end, freq='15min', tz='Europe/London')
        if len(dr):
            miss = pd.DataFrame({'time': dr})
            h = (miss['time'] - df['time'].min()).dt.total_seconds() / 3600
            miss['level'] = sin_function(h.values, a, b, c, d)
            df = pd.concat([df, miss], ignore_index=True).sort_values('time').reset_index(drop=True)
        df['level_cm'] = (df['level'] * 100).round()
        df['diff_cm'] = df['level_cm'].diff().abs()
        df['payoff'] = np.maximum(0, 20 - df['diff_cm']) + np.maximum(0, df['diff_cm'] - 25)
        fv = float(df['payoff'].sum())
        # 不确定性：振幅误差传播
        hours = (df['time'] - df['time'].min()).dt.total_seconds() / 3600
        residuals = df['level'].values - sin_function(hours.values, a, b, c, d)
        std = float(np.std(residuals)) * 100 * len(df) ** 0.5
        return round(fv), max(std, 1.0)

    def lhr_count_fv(self):
        if self._flights_cache is None:
            return 1400, 50.0  # 历史均值先验
        d1, d2 = self._flights_cache
        fv = float(get_lhr_count_from(d1, d2))
        return round(fv), 30.0  # 航班数方差较小

    def lhr_index_fv(self):
        if self._flights_cache is None:
            return 50, 20.0
        d1, d2 = self._flights_cache
        fv = float(get_lhr_index_from(d1, d2))
        return round(fv), 15.0

    def lon_etf_fv(self):
        ts, ts_std = self.tide_spot_fv()
        wx, wx_std = self.wx_spot_fv()
        lc, lc_std = self.lhr_count_fv()
        fv = ts + wx + lc
        std = (ts_std**2 + wx_std**2 + lc_std**2) ** 0.5
        return round(fv), max(std, 1.0)

    def lon_fly_fv(self):
        etf, etf_std = self.lon_etf_fv()
        def fly_payoff(e):
            return (2*max(6200-e,0) + max(e-6200,0)
                    - 2*max(e-6600,0) + 3*max(e-7000,0))
        fv = fly_payoff(etf)
        # delta 用数值微分
        delta = (fly_payoff(etf+1) - fly_payoff(etf-1)) / 2
        std = abs(delta) * etf_std
        return round(fv), max(std, 1.0)

    def get_all(self):
        """返回所有产品的 {symbol: (fv, std)}"""
        return {
            'TIDE_SPOT':  self.tide_spot_fv(),
            'TIDE_SWING': self.tide_swing_fv(),
            'WX_SPOT':    self.wx_spot_fv(),
            'WX_SUM':     self.wx_sum_fv(),
            'LHR_COUNT':  self.lhr_count_fv(),
            'LHR_INDEX':  self.lhr_index_fv(),
            'LON_ETF':    self.lon_etf_fv(),
            'LON_FLY':    self.lon_fly_fv(),
        }


In [ ]:
def get_lhr_count_from(d1, d2):
    return (len(d1.get('arrivals',[])) + len(d1.get('departures',[]))
            + len(d2.get('arrivals',[])) + len(d2.get('departures',[])))

def get_lhr_index_from(d1, d2):
    date_range = pd.date_range(
        start=LONDON_TZ.localize(datetime(2026,2,28,12,0,0)),
        end=LONDON_TZ.localize(datetime(2026,3,1,12,0,0)),
        freq='30min', tz='Europe/London')
    idx = 0
    for i in range(24):
        arr, dep = group_flights(d1, date_range[i], date_range[i+1])
        idx += ((arr-dep)/(arr+dep))*100 if arr+dep else 0
    for i in range(24):
        arr, dep = group_flights(d2, date_range[i+24], date_range[i+25])
        idx += ((arr-dep)/(arr+dep))*100 if arr+dep else 0
    return round(abs(idx))


In [ ]:
class AlgoBot(BaseBot):
    """
    三模块策略:
      A. Fair-value 做市 (主要收益)
      B. ETF 成分套利 (无风险)
      C. 方向性押注 (z-score 超阈值时)
    """
    SYMBOLS = ['TIDE_SPOT','TIDE_SWING','WX_SPOT','WX_SUM',
               'LHR_COUNT','LHR_INDEX','LON_ETF','LON_FLY']
    ETF_LEGS = ['TIDE_SPOT','WX_SPOT','LHR_COUNT']
    POSITION_LIMIT = 100

    # 策略参数
    MM_SPREAD_MULTIPLIER = 1.5   # spread = std * multiplier
    MM_MIN_SPREAD = 3            # 最小 spread（tick 单位）
    MM_BASE_QTY = 10
    ARB_THRESHOLD = 10           # 套利触发阈值（tick）
    ARB_QTY = 20
    DIR_Z_THRESHOLD = 2.0        # 方向性触发 z-score
    DIR_QTY = 15
    REQUOTE_THRESHOLD = 2        # fair value 变化超过此值才重挂单
    LOOP_INTERVAL = 10           # 主循环间隔（秒）

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.fve = FairValueEngine()
        self.fair_values = {}     # {symbol: (fv, std)}
        self.last_quotes = {}     # {symbol: (bid_price, ask_price)}
        self.active_orders = {}   # {symbol: [order_id, ...]}

    def on_orderbook(self, ob): pass
    def on_trades(self, trade):
        side = 'BOUGHT' if trade.buyer == self.username else 'SOLD'
        print(f'  FILL {side} {trade.volume}x {trade.product} @ {trade.price}')

    # ── 工具方法 ──────────────────────────────────────────────────────────────
    def _get_mid(self, ob):
        """从 orderbook 取 mid，排除自己的订单"""
        bids = [o.price for o in ob.buy_orders if o.volume - o.own_volume > 0]
        asks = [o.price for o in ob.sell_orders if o.volume - o.own_volume > 0]
        if bids and asks:
            return (max(bids) + min(asks)) / 2
        return None

    def _send_ioc(self, order):
        resp = self.send_order(order)
        if resp and resp.status == 'ACTIVE':
            self.cancel_order(resp.id)
        return resp

    def _cancel_symbol_orders(self, symbol):
        for oid in self.active_orders.get(symbol, []):
            self.cancel_order(oid)
        self.active_orders[symbol] = []

    def _position_skew(self, position, fv, std):
        """根据仓位偏斜 bid/ask，仓位越重偏斜越大"""
        ratio = position / self.POSITION_LIMIT
        skew = ratio * std * 0.5
        return skew

    # ── 模块 A：Fair-value 做市 ───────────────────────────────────────────────
    def module_a_market_make(self, positions):
        orders_placed = 0
        for symbol in self.SYMBOLS:
            if symbol not in self.fair_values:
                continue
            fv, std = self.fair_values[symbol]
            ob = self.get_orderbook(symbol)
            market_mid = self._get_mid(ob)

            # 用 fair value 作为报价中心，market mid 作为参考
            quote_mid = fv
            half_spread = max(self.MM_MIN_SPREAD, std * self.MM_SPREAD_MULTIPLIER / 2)

            # 仓位偏斜
            pos = positions.get(symbol, 0)
            skew = self._position_skew(pos, fv, std)
            bid_price = round(quote_mid - half_spread - skew)
            ask_price = round(quote_mid + half_spread - skew)

            if bid_price <= 0 or ask_price <= bid_price:
                continue

            # 只有 fair value 变化超过阈值才重挂（保留队列优先级）
            last = self.last_quotes.get(symbol)
            if last and abs(last[0] - bid_price) < self.REQUOTE_THRESHOLD \
                    and abs(last[1] - ask_price) < self.REQUOTE_THRESHOLD:
                continue

            self._cancel_symbol_orders(symbol)

            avail_buy = self.POSITION_LIMIT - pos
            avail_sell = self.POSITION_LIMIT + pos
            bid_qty = max(1, min(self.MM_BASE_QTY, avail_buy))
            ask_qty = max(1, min(self.MM_BASE_QTY, avail_sell))

            new_ids = []
            if avail_buy >= 1:
                r = self.send_order(OrderRequest(symbol, bid_price, Side.BUY, bid_qty))
                if r: new_ids.append(r.id)
            if avail_sell >= 1:
                r = self.send_order(OrderRequest(symbol, ask_price, Side.SELL, ask_qty))
                if r: new_ids.append(r.id)

            self.active_orders[symbol] = new_ids
            self.last_quotes[symbol] = (bid_price, ask_price)
            orders_placed += 1
        return orders_placed

    # ── 模块 B：ETF 成分套利 ──────────────────────────────────────────────────
    def module_b_arbitrage(self, positions):
        obs = {s: self.get_orderbook(s) for s in ['LON_ETF'] + self.ETF_LEGS}
        mids = {s: self._get_mid(obs[s]) for s in obs}
        if any(v is None for v in mids.values()):
            return

        etf_mid = mids['LON_ETF']
        comp_sum = sum(mids[s] for s in self.ETF_LEGS)
        basis = etf_mid - comp_sum

        if basis > self.ARB_THRESHOLD:
            # ETF 贵：卖 ETF，买三个成分
            print(f'  ARB: ETF overpriced by {basis:.1f}, selling ETF buying legs')
            qty = min(self.ARB_QTY,
                      self.POSITION_LIMIT + positions.get('LON_ETF', 0),
                      *[self.POSITION_LIMIT - positions.get(s, 0) for s in self.ETF_LEGS])
            if qty >= 1:
                etf_ask = min(o.price for o in obs['LON_ETF'].sell_orders) if obs['LON_ETF'].sell_orders else None
                if etf_ask:
                    self._send_ioc(OrderRequest('LON_ETF', etf_ask, Side.SELL, qty))
                for s in self.ETF_LEGS:
                    best_ask = min(o.price for o in obs[s].sell_orders) if obs[s].sell_orders else None
                    if best_ask:
                        self._send_ioc(OrderRequest(s, best_ask, Side.BUY, qty))

        elif basis < -self.ARB_THRESHOLD:
            # ETF 便宜：买 ETF，卖三个成分
            print(f'  ARB: ETF underpriced by {abs(basis):.1f}, buying ETF selling legs')
            qty = min(self.ARB_QTY,
                      self.POSITION_LIMIT - positions.get('LON_ETF', 0),
                      *[self.POSITION_LIMIT + positions.get(s, 0) for s in self.ETF_LEGS])
            if qty >= 1:
                best_bid = max(o.price for o in obs['LON_ETF'].buy_orders) if obs['LON_ETF'].buy_orders else None
                if best_bid:
                    self._send_ioc(OrderRequest('LON_ETF', best_bid, Side.BUY, qty))
                for s in self.ETF_LEGS:
                    best_bid_s = max(o.price for o in obs[s].buy_orders) if obs[s].buy_orders else None
                    if best_bid_s:
                        self._send_ioc(OrderRequest(s, best_bid_s, Side.SELL, qty))

    # ── 模块 C：方向性押注 ────────────────────────────────────────────────────
    def module_c_directional(self, positions):
        for symbol in self.SYMBOLS:
            if symbol not in self.fair_values:
                continue
            fv, std = self.fair_values[symbol]
            if std <= 0:
                continue
            ob = self.get_orderbook(symbol)
            market_mid = self._get_mid(ob)
            if market_mid is None:
                continue

            z = (market_mid - fv) / std
            pos = positions.get(symbol, 0)

            if z > self.DIR_Z_THRESHOLD:
                # 市场高估，做空
                qty = min(self.DIR_QTY, self.POSITION_LIMIT + pos)
                if qty >= 1 and ob.buy_orders:
                    best_bid = max(o.price for o in ob.buy_orders)
                    print(f'  DIR SELL {symbol} z={z:.2f} fv={fv} mid={market_mid:.0f}')
                    self._send_ioc(OrderRequest(symbol, best_bid, Side.SELL, qty))

            elif z < -self.DIR_Z_THRESHOLD:
                # 市场低估，做多
                qty = min(self.DIR_QTY, self.POSITION_LIMIT - pos)
                if qty >= 1 and ob.sell_orders:
                    best_ask = min(o.price for o in ob.sell_orders)
                    print(f'  DIR BUY  {symbol} z={z:.2f} fv={fv} mid={market_mid:.0f}')
                    self._send_ioc(OrderRequest(symbol, best_ask, Side.BUY, qty))

    # ── LON_FLY delta 对冲 ────────────────────────────────────────────────────
    def module_d_fly_hedge(self, positions):
        fly_pos = positions.get('LON_FLY', 0)
        if fly_pos == 0:
            return
        etf_mid = self._get_mid(self.get_orderbook('LON_ETF'))
        if etf_mid is None:
            return
        # FLY delta w.r.t. ETF
        if etf_mid < 6200:   fly_delta = -2
        elif etf_mid < 6600: fly_delta = -1
        elif etf_mid < 7000: fly_delta =  1
        else:                fly_delta = -2
        target_etf_hedge = -fly_pos * fly_delta
        current_etf_pos = positions.get('LON_ETF', 0)
        delta_needed = target_etf_hedge - current_etf_pos
        if abs(delta_needed) < 5:
            return
        ob = self.get_orderbook('LON_ETF')
        qty = min(abs(delta_needed), self.POSITION_LIMIT - abs(current_etf_pos))
        if qty < 1:
            return
        if delta_needed > 0 and ob.sell_orders:
            best_ask = min(o.price for o in ob.sell_orders)
            print(f'  HEDGE BUY LON_ETF {qty} (fly_delta={fly_delta})')
            self._send_ioc(OrderRequest('LON_ETF', best_ask, Side.BUY, qty))
        elif delta_needed < 0 and ob.buy_orders:
            best_bid = max(o.price for o in ob.buy_orders)
            print(f'  HEDGE SELL LON_ETF {qty} (fly_delta={fly_delta})')
            self._send_ioc(OrderRequest('LON_ETF', best_bid, Side.SELL, qty))

    # ── 主循环 ────────────────────────────────────────────────────────────────
    def run_bot(self):
        print('Initialising flights data...')
        self.fve.init_flights()
        self.start()  # 启动 SSE
        print('Bot started. Press Ctrl+C to stop.')
        try:
            while True:
                t0 = time.time()

                # 1. 更新 fair values
                self.fair_values = self.fve.get_all()
                fv_summary = {s: f'{v[0]}±{v[1]:.0f}' for s, v in self.fair_values.items()}
                print(f'\n[{datetime.now().strftime("%H:%M:%S")}] FV: {fv_summary}')

                # 2. 获取仓位
                positions = self.get_positions()
                print(f'  Positions: {positions}')

                # 3. 套利优先（无风险，先执行）
                self.module_b_arbitrage(positions)

                # 4. 方向性押注
                self.module_c_directional(positions)

                # 5. FLY delta 对冲
                self.module_d_fly_hedge(positions)

                # 6. 做市报价
                self.module_a_market_make(positions)

                elapsed = time.time() - t0
                sleep_time = max(1, self.LOOP_INTERVAL - elapsed)
                time.sleep(sleep_time)

        except KeyboardInterrupt:
            print('\nStopping bot...')
            self.cancel_all_orders()
            self.stop()
            print('Bot stopped. Final positions:', self.get_positions())
        except Exception as e:
            import traceback
            print(f'Error: {e}')
            traceback.print_exc()
            self.cancel_all_orders()
            self.stop()


In [ ]:
bot = AlgoBot(EXCHANGE_URL, USERNAME, PASSWORD)
bot.run_bot()
